# 01 — Datos originales: entender antes de modelar

## Pregunta que guía la caso

Queremos ordenar clientes por probabilidad de volver a comprar en los 30 días posteriores a un corte. Antes de entrenar hay que responder algo más básico: qué representa una fila y qué registros pueden formar parte de un perfil de cliente.

**Resultado de este notebook:** diagnóstico reproducible del archivo original. Aquí todavía no se elimina ni se corrige ningún dato.

In [ ]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')
def find_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path('/content/retail_customer_intelligence')]:
        if (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError('No se encontró la raíz del proyecto. Abre la carpeta retail_customer_intelligence, no sólo el archivo .ipynb.')
ROOT = find_root()
sys.path.insert(0, str(ROOT / 'src'))
from retail_pipeline import *
ensure_folders(ROOT)
print('Proyecto:', ROOT)

## Contrato de datos: qué pocasos y qué no pocasos inferir

El archivo representa líneas de factura, no clientes ni pedidos completos. Una factura puede tener varios artículos y un cliente puede aparecer muchas veces. Si se agregara directamente sin comprender esta granularidad, se duplicaría o se atribuiría gasto de manera equivocada.

In [ ]:
raw, downloaded = load_original(ROOT)
print("Descarga nueva en esta ejecución:", downloaded)
print("Filas y columnas:", raw.shape)
print("Periodo:", raw.InvoiceDate.min().date(), "a", raw.InvoiceDate.max().date())
contract = pd.DataFrame({
    "campo": raw.columns,
    "tipo": raw.dtypes.astype(str),
    "nulos": raw.isna().sum().to_numpy(),
    "ejemplo": [
        raw[column].dropna().iloc[0] if raw[column].notna().any() else None
        for column in raw.columns
    ],
})
display(contract)
print("Una observación para modelar no existirá hasta agregar estas líneas por cliente y fecha de corte.")

## Interpretar los problemas de calidad antes de descartarlos

Un valor raro no es necesariamente un error. Una cancelación, una devolución o una línea sin cliente describen hechos distintos. La siguiente tabla convierte la inspección en decisiones que se podrán auditar en el siguiente notebook.

In [ ]:
states = raw.assign(estado=np.select(
    [
        raw.InvoiceNo.astype("string").str.startswith("C", na=False),
        raw.Quantity.lt(0),
        raw.UnitPrice.le(0),
        raw.CustomerID.isna(),
    ],
    ["cancelación", "devolución/ajuste", "precio no positivo", "sin CustomerID"],
    default="venta candidata",
)).groupby("estado").agg(
    líneas=("InvoiceNo", "size"),
    facturas=("InvoiceNo", "nunique"),
    clientes=("CustomerID", "nunique"),
).sort_values("líneas", ascending=False)
display(states)

data_contract = pd.DataFrame([
    ("venta candidata", "Puede contribuir al perfil de compra", "validar antes de usar"),
    ("cancelación o devolución", "Evidencia de ajuste, no de compra", "no sumar como venta"),
    ("sin CustomerID", "No se puede asociar a una persona", "no usar en modelo por cliente"),
], columns=["tipo de fila", "significado para la pregunta", "decisión posterior"])
display(data_contract)

## Por qué respetaremos el tiempo

La actividad no es constante a lo largo de los meses. Una evaluación aleatoria puede mezclar una temporada posterior con el entrenamiento y crear una impresión optimista. La prueba final se reservará en el futuro, como ocurriría al lanzar una campaña.

In [ ]:
valid = raw.loc[
    ~raw.InvoiceNo.astype("string").str.startswith("C", na=False)
    & raw.CustomerID.notna()
    & raw.Quantity.gt(0)
    & raw.UnitPrice.gt(0)
].copy()
valid["importe_línea"] = valid.Quantity * valid.UnitPrice
valid["mes"] = valid.InvoiceDate.dt.to_period("M").astype(str)
monthly_check = valid.groupby("mes").agg(
    ventas=("importe_línea", "sum"),
    facturas=("InvoiceNo", "nunique"),
    clientes=("CustomerID", "nunique"),
)
display(monthly_check)
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
monthly_check.ventas.plot(ax=axes[0], marker="o", color="#2a9d8f", title="Ventas válidas por mes")
monthly_check[["facturas", "clientes"]].plot(ax=axes[1], marker="o", title="Actividad por mes")
plt.tight_layout()

## Mensaje para clase

Antes del primer modelo ya aprendimos algo importante: el archivo no está listo para modelar porque sus filas tienen significados y granularidades diferentes. El producto de esta fase es evidencia para decidir reglas, no una gráfica decorativa.

Artefacto: reports/diagnostico_datos_originales.csv.